In [200]:
import matplotlib as mpl
from math import sqrt
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image 
from scipy.stats import kde 
import seaborn as sns
import pandas as pd
import xlwt
import xlrd
import openpyxl
import sklearn
import mca
import prince
from statsmodels.stats.multicomp import MultiComparison
import statsmodels.stats.proportion as stats
from scipy.stats import chi2_contingency
from matplotlib import style
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [164]:
import pandas as pd
import numpy as np
from datetime import datetime

# Carga de datos
datos_total = pd.read_excel('Consolidado.xlsx')
datos_total.columns = [col.strip() for col in datos_total.columns]

# Cálculo de variables faltantes
birth_col = "TO_CHAR(E.FECHANACIMIENTO,'DD-MM-YYYY')"
if birth_col not in datos_total.columns and birth_col + ' ' in datos_total.columns:
    birth_col = birth_col + ' '

datos_total['birth_date'] = pd.to_datetime(datos_total[birth_col], unit='D', origin='1899-12-30')
reference_date = pd.to_datetime('2022-01-01')
datos_total['EDAD'] = datos_total['birth_date'].apply(lambda x: reference_date.year - x.year - ((reference_date.month, reference_date.day) < (x.month, x.day)))
datos_total['FEMENINO'] = datos_total['SEXO'].apply(lambda x: 1 if str(x).strip().upper() == 'F' else 0)

bins = [0, 18, 25, 35, 45, 55, 65, 120]
labels = ['0-18', '19-25', '26-35', '36-45', '46-55', '56-65', '66+']
datos_total['RANGO DE EDAD'] = pd.cut(datos_total['EDAD'], bins=bins, labels=labels)

if 'INICIO_OCURRENCIA' in datos_total.columns:
    datos_total['FECHA INICIAL'] = datos_total['INICIO_OCURRENCIA']

print('Variables preprocesadas correctamente.')

In [165]:
datos_total.head()

In [166]:
datos_total.info()

In [167]:
tabla_contingencia = pd.crosstab(datos_total['EDAD'], [datos_total['TIPO'], datos_total['SUBTIPO'], datos_total['FEMENINO']], margins=True)

In [168]:
tabla_contingencia

In [169]:
#Tabla de contingencia relativa
tabla_contingencia_r = pd.crosstab(index=datos_total['EDAD'], columns=([datos_total['TIPO'], datos_total['SUBTIPO'], datos_total['FEMENINO']]),
            margins=True).apply(lambda r: r/len(datos_total) *100,
                                axis=1)
tabla_contingencia_r

In [170]:
# Utilizamos la función chi2_contingency para calcular el índice de Chi-cuadrado y el p-valor
chi2, p, dof, expected = chi2_contingency(tabla_contingencia.drop('All', axis=0).drop('All', axis=1, level=0))
# Si el p-valor es menor que 0.05, podemos decir que existe una relación significativa entre las variables
if p < 0.05:
    print("Existe una relación significativa entre las variables (p = {})".format(p))
else:
    print("No existe una relación significativa entre las variables (p = {})".format(p))

In [171]:
# resultados_acm = stats.multinomial_proportions_confint(tabla_contingencia)
 # Removed incorrect stats call

# Muestra los resultados
print(resultados_acm)

In [174]:
# Carga los resultados del ACM
matriz_cargas = resultados_acm[0][0]
varianza_explicada = resultados_acm[0][1]

# Crea el gráfico de barras
plt.bar(range(len(varianza_explicada)), varianza_explicada)
plt.xlabel("Componente principal")
plt.ylabel("Varianza explicada")
plt.title("Análisis de correspondencias múltiples")
plt.show()


In [185]:


# Carga los resultados del ACM
varianza_explicada = resultados_acm[0][1]

# Crea el gráfico de líneas
plt.plot(range(len(varianza_explicada)), varianza_explicada)
plt.xlabel("Componente principal")
plt.ylabel("Varianza explicada")
plt.title("Análisis de correspondencias múltiples")
plt.show()

In [189]:


# Carga los resultados del ACM
matriz_cargas = resultados_acm[0][0]

# Crea el gráfico de dispersión
plt.scatter(range(len(varianza_explicada)), varianza_explicada)
plt.xlabel("Componente 1")
plt.ylabel("Componente 2")
plt.title("Plano factorial del análisis de correspondencias múltiples")
plt.show()


In [197]:


# Calcula el análisis de correspondencias múltiples
# resultados_acm = stats.multinomial_proportions_confint(tabla_contingencia)
 # Removed incorrect stats call

# Accede a los resultados del ACM
matriz_cargas = resultados_acm[0][0]  # Matriz de cargas
varianza_explicada = resultados_acm[0][1]  # Varianza explicada
proporciones = resultados_acm[0][2]  # Proporciones

# Muestra los resultados
print(matriz_cargas)
print(varianza_explicada)
print(proporciones)

In [180]:
# Calcula el indicador de asociación Phi de Pearson
a = tabla_contingencia.iloc[0, 0]
b = tabla_contingencia.iloc[0, 1]
c = tabla_contingencia.iloc[0, 2]
d = tabla_contingencia.iloc[0, 3]

try:
   # Calcula el indicador de asociación Phi de Pearson
   phi = (a * d - b * c) / ((a + b) * (c + d))**0.5
except ZeroDivisionError:
   # Maneja la excepción de división por cero
   print("Se ha producido una división por cero")
phi = None

In [149]:
# Calcula el indicador de asociación Cramer's V
n = tabla_contingencia.loc['All', 'All']
k = tabla_contingencia.shape[0]

    import numpy as np
    min_dim = min(tabla_contingencia.drop('All', axis=0).shape[0], tabla_contingencia.drop('All', axis=1, level=0).shape[1])
    cramers = np.sqrt(chi2 / (n * min_dim))


In [160]:
# Crea un objeto MCA
mca_obj = mca.MCA(tabla_contingencia.drop('All', axis=0).drop('All', axis=1, level=0))

# Calcula el análisis factorial y los componentes principales
mca_results = mca_obj.fs_r()

# Accede a los resultados del análisis
print(mca_results.L)  # Matriz de cargas
print(mca_results.expl_var)  # Varianza explicada

In [118]:
n = tabla_contingencia.loc['All', 'All']
k = tabla_contingencia.shape[0]

    import numpy as np
    min_dim = min(tabla_contingencia.drop('All', axis=0).shape[0], tabla_contingencia.drop('All', axis=1, level=0).shape[1])
    cramers = np.sqrt(chi2 / (n * min_dim))


In [ ]:
# Calcula el indicador de asociación Cramer's V
n = tabla_contingencia.loc['All', 'All']
k = tabla_contingencia.shape[0]

    import numpy as np
    min_dim = min(tabla_contingencia.drop('All', axis=0).shape[0], tabla_contingencia.drop('All', axis=1, level=0).shape[1])
    cramers = np.sqrt(chi2 / (n * min_dim))


In [117]:
# Crea un objeto MCA
mca_obj = mca.MCA(tabla_contingencia.drop('All', axis=0).drop('All', axis=1, level=0))

try:
   # Calcula el análisis factorial y los componentes principales
   mca_results = mca_obj.fs_r()
except ZeroDivisionError:
   # Maneja la excepción de división por cero
   print("Se ha producido una división por cero")
   mca_results = None

# Accede a los resultados del análisis
print(mca_results.L)  # Matriz
print(mca_results.expl_var)  # Varianza explicada

In [114]:
# Crea un objeto MCA
mca_obj = mca.MCA(tabla_contingencia.drop('All', axis=0).drop('All', axis=1, level=0))
try:
   # Calcula el análisis factorial y los componentes principales
   mca_results = mca_obj.fs_r()
except ZeroDivisionError:
   # Maneja la excepción de división por cero
   print("Se ha producido una división por cero")
mca_results = None
print(mca_results.L)  # Matriz de cargas
print(mca_results.expl_var)  # Varianza explicada

In [115]:
# Crea un objeto MCA
mca_obj = mca.MCA(tabla_contingencia.drop('All', axis=0).drop('All', axis=1, level=0))

# Calcula el análisis factorial y los componentes principales
mca_results = mca_obj.fs_r()

# Accede a los resultados del análisis
print(mca_results.L)  # Matriz de cargas
print(mca_results.expl_var)  # Varianza explicada

In [98]:
# Crea un objeto MCA
mca_obj = mca.MCA(tabla_contingencia.drop('All', axis=0).drop('All', axis=1, level=0))
try:
   # Calcula el análisis factorial y los componentes principales
   mca_results = mca_obj.fs_r()
except ZeroDivisionError:
   # Maneja la excepción de división por cero
   print("Se ha producido una división por cero")
   mca_results = None

In [ ]:
dat=datos_total.to_numpy()

In [19]:
# Visulazing the distibution of the data for every feature
tabla_contingencia.hist(edgecolor='black', linewidth=3, figsize=(50, 50));

In [27]:
#graficando 
plot = datos_total['SUBTIPO'].plot(figsize=(200, 50))

In [34]:
#Establecemos los límites del eje x y y
plt.xlim(-1, 1)
plt.ylim(-1, 1.3)

# Dibujamos el gráfico de líneas
plt.plot(acm)

# Mostramos el gráfico
plt.show()

In [ ]:
datos2 = datos_total[['FECHA INICIAL', 'TIPO', 'SUBTIPO', 'EDAD', 'FEMENINO', 'RANGO DE EDAD']]

In [ ]:
mca = prince.MCA()
mca.fit(datos2.drop('FECHA INICIAL', axis=1).astype(str))

In [ ]:
mca.plot_coordinates(datos2,
                     row_points_alpha=.2,
                     figsize=(10, 10),
                     show_column_labels=True
                    );

In [ ]:
#graficando 
plot = datos_total['EDAD'].plot(figsize=(200, 200))

In [ ]:

mca = prince.MCA(tabla_contingencia)
mca.fit(tabla_contingencia)

In [ ]:
tabla_contingencia2 = pd.crosstab(datos_total['FECHA INICIAL'], [datos_total['EDAD'], datos_total['FEMENINO'], datos_total['RANGO DE EDAD']], margins=True)
tabla_contingencia2

In [ ]:
tabla_contingencia.describe()